# EfficientDet 실습 — Colab 실행용

> **실행 전 꼭!** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 `런타임 → 모두 실행`
> GPU 없이도 돌아가지만 학습이 매우 느립니다.

이 노트북은 **파일을 따로 업로드하지 않아도** 처음부터 끝까지 실행되도록 원본을 수정한 버전입니다.
수정한 셀에는 `# ✏️ 수정:` 주석을 달아 두었습니다.

| 원래 문제 | 수정 내용 |
|---|---|
| `soccer.mp4`를 직접 올려야 함 | 이 저장소에 올려 둔 `soccer.mp4`를 자동으로 내려받음 (원본은 437MB GIF라 GitHub 용량 제한 때문에 1280px mp4로 변환 — 원본과 해상도가 다른 건 어쩔 수 없었음. 모델 입력은 어차피 512px로 줄어들어 결과 차이는 거의 없음) |
| `archive.zip`을 직접 올려야 함 | Kaggle에서 자동 다운로드 (로그인 불필요) |
| `projects/my_car_detect_proj.yml`을 직접 만들어야 함 | 계산한 mean/std·앵커·클래스로 **자동 생성** |
| `train.py`의 `verbose=True` 에러를 손으로 고쳐야 함 | `sed`로 자동 수정 |
| 가중치 다운로드 URL 오타 | 올바른 저장소 주소로 수정 |
| 셀을 다시 실행하면 clone·chdir·경로 에러 | 다시 실행해도 안전하게 수정 |
| `use_cuda = True` 고정 | GPU 유무 자동 판단 |
| 학습 데이터(흑백 v9i) ≠ 분석·테스트 데이터(컬러 v8i) | **컬러 v8i로 통일** |
| 학습·추론 앵커 불일치 | 추론도 yml 값 사용 |
| `obj_list`에 박스 없는 `'cars'` 포함 → 이름 한 칸 밀림 | 실제 5개 클래스만 사용 |
| `num_gpus: 0` → GPU가 있어도 CPU로 학습 | GPU 있으면 `num_gpus: 1` |

## 1. 라이브러리 설치 및 환경 설정

In [ ]:
# ✏️ 수정: 이미 clone 되어 있으면 건너뜀 (다시 실행해도 에러 없음)
import os
if not os.path.exists('/content/Yet-Another-EfficientDet-Pytorch'):
    !git clone https://github.com/zylo117/Yet-Another-EfficientDet-Pytorch.git /content/Yet-Another-EfficientDet-Pytorch

# ✏️ 수정: 학습·평가에 필요한 패키지를 미리 설치
!pip install -q pycocotools tensorboardX

In [ ]:
import os

# ✏️ 수정: 절대 경로로 이동 → 몇 번을 다시 실행해도 같은 위치
os.chdir('/content/Yet-Another-EfficientDet-Pytorch')
print("현재 디렉토리:", os.getcwd())

In [ ]:
# ✏️ 수정(추가): GPU 사용 가능 여부 확인 → 아래 셀들의 use_cuda에 사용
import torch
print("GPU 사용 가능:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️ GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 권장합니다.")

In [ ]:
!mkdir -p weights
# ✏️ 수정: URL 저장소 이름 오타 수정 (Efficient → EfficientDet), -nc: 이미 있으면 다시 받지 않음
!wget -nc -q https://github.com/zylo117/Yet-Another-EfficientDet-Pytorch/releases/download/1.0/efficientdet-d0.pth -O weights/efficientdet-d0.pth
!ls -lh weights

In [ ]:
import torch
from backbone import EfficientDetBackbone
compound_coef = 0
model = EfficientDetBackbone(compound_coef=0, num_classes=90)

In [ ]:
model.load_state_dict(torch.load(f'weights/efficientdet-d0.pth', map_location='cpu'))
model.requires_grad_(False)
model.eval()

In [ ]:
compound_coef = 0
force_input_size = None  # set None to use default size
# img_path = 'bus.jpg'

# replace this part with your project's anchor config
anchor_ratios = [(1.0, 1.0), (1.4, 0.7), (0.7, 1.4)]
anchor_scales = [2 ** 0, 2 ** (1.0 / 3.0), 2 ** (2.0 / 3.0)]

threshold = 0.2
iou_threshold = 0.2

use_cuda = torch.cuda.is_available()  # ✏️ 수정: GPU 유무 자동 판단
use_float16 = False

In [ ]:
obj_list = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
            'fire hydrant', '', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep',
            'cow', 'elephant', 'bear', 'zebra', 'giraffe', '', 'backpack', 'umbrella', '', '', 'handbag', 'tie',
            'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
            'skateboard', 'surfboard', 'tennis racket', 'bottle', '', 'wine glass', 'cup', 'fork', 'knife', 'spoon',
            'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut',
            'cake', 'chair', 'couch', 'potted plant', 'bed', '', 'dining table', '', '', 'toilet', '', 'tv',
            'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
            'refrigerator', '', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
            'toothbrush']

In [ ]:
len(obj_list)

In [ ]:
from PIL import Image
Image.open('test/img.png')

In [ ]:
from utils.utils import preprocess, invert_affine, postprocess, STANDARD_COLORS, standard_to_bgr, get_index_label, plot_one_box

img_path = 'test/img.png'
color_list = standard_to_bgr(STANDARD_COLORS)
input_sizes = [512, 640, 768, 896, 1024, 1280, 1280, 1536, 1536]
input_size = input_sizes[compound_coef] if force_input_size is None else force_input_size
ori_imgs, framed_imgs, framed_metas = preprocess(img_path, max_size=input_size)
if use_cuda:
    x = torch.stack([torch.from_numpy(fi).cuda() for fi in framed_imgs], 0)
else:
    x = torch.stack([torch.from_numpy(fi) for fi in framed_imgs], 0)
x = x.to(torch.float32 if not use_float16 else torch.float16).permute(0, 3, 1, 2)

In [ ]:
ori_imgs[0].shape, framed_imgs[0].shape

In [ ]:
framed_metas

In [ ]:
if use_cuda:
    model = model.cuda()
if use_float16:
    model = model.half()
with torch.no_grad():
    features, regression, classification, anchors = model(x)

In [ ]:
features[0].shape, regression[0].shape, classification.shape, anchors.shape

In [ ]:
len(features)

In [ ]:
features[0].shape, features[1].shape, features[2].shape, features[3].shape, features[4].shape

In [ ]:
from utils.utils import postprocess
from efficientdet.utils import BBoxTransform, ClipBoxes

with torch.no_grad():
    features, regression, classification, anchors = model(x)
    regressBoxes = BBoxTransform() # 모델이 예측한 regression 값을 실제 bounding box 좌표로 변환
    clipBoxes = ClipBoxes()        # 모델이 예측한 object의 box가 이미지 범위를 벗어나지 않도록 조정
    out = postprocess(x,
                      anchors, regression, classification,
                      regressBoxes, clipBoxes,
                      threshold, iou_threshold)

In [ ]:
out

In [ ]:
out[0]['rois'].shape, out[0]['class_ids'].shape, out[0]['scores'].shape

In [ ]:
import cv2
import numpy as np  # ✏️ 수정: 함수 안에서 쓰는 np를 미리 import
def display(preds, imgs, imshow=True, imwrite=False):
    for i in range(len(imgs)):
        if len(preds[i]['rois']) == 0:
            continue

        imgs[i] = imgs[i].copy()

        for j in range(len(preds[i]['rois'])):
            x1, y1, x2, y2 = preds[i]['rois'][j].astype(np.int32)
            obj = obj_list[preds[i]['class_ids'][j]]
            score = float(preds[i]['scores'][j])
            plot_one_box(imgs[i], [x1, y1, x2, y2], label=obj,score=score,color=color_list[get_index_label(obj, obj_list)])


        if imshow:
            cv2.imshow('img', imgs[i])
            cv2.waitKey(0)

        if imwrite:
            cv2.imwrite(f'test/img_inferred_d{compound_coef}_this_repo_{i}.jpg', imgs[i])

    return f'test/img_inferred_d{compound_coef}_this_repo_'

In [ ]:
import numpy as np
out = invert_affine(framed_metas, out)
img_path = display(out, ori_imgs, imshow=False, imwrite=True)

In [ ]:
img_path

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# ✏️ 수정: img_path를 덮어쓰면 이 셀을 다시 실행할 때 경로가 ..0.jpg0.jpg 가 되어 에러 → 새 변수 사용
result_img_path = f"{img_path}0.jpg"

img = Image.open(result_img_path)

# 이미지 출력
plt.imshow(img)
plt.axis('off')  # 축 제거
plt.title(result_img_path)
plt.show()

### 영상 분석

In [ ]:
# ✏️ 수정: soccer.mp4를 직접 올리지 않아도 GitHub 저장소에서 자동으로 내려받음
#          원본은 437MB GIF라 GitHub 용량 제한(100MB) 때문에 1280px mp4로 변환해서 올림
#          → 원본과 해상도가 다른 건 어쩔 수 없었음 (모델 입력은 어차피 512px로 줄어들어 결과 차이는 거의 없음)
video_src = 'soccer.mp4'
if not os.path.exists(video_src):
    !wget -q https://github.com/hwang-ye-song/EfficientDet-_CV/raw/main/soccer.mp4 -O soccer.mp4
# 그래도 받지 못하면 예비로 OpenCV 공식 샘플 영상(보행자)을 사용
if not os.path.exists(video_src) or os.path.getsize(video_src) == 0:
    !wget -nc -q https://github.com/opencv/opencv/raw/4.x/samples/data/vtest.avi
    video_src = 'vtest.avi'
print("사용할 영상:", video_src, os.path.exists(video_src))

In [ ]:
# Core Author: Zylo117
# Script's Author: winter2897

"""
Simple Inference Script of EfficientDet-Pytorch for detecting objects on webcam
"""
import time
import torch
import cv2
import numpy as np
from torch.backends import cudnn
from backbone import EfficientDetBackbone
from efficientdet.utils import BBoxTransform, ClipBoxes
from utils.utils import preprocess, invert_affine, postprocess, preprocess_video

# Video's path
# ✏️ 수정: video_src는 바로 위 셀에서 정함 (여기서 다시 'soccer.mp4'로 덮어쓰지 않음)
# output_dir = './output_soccer_frames'
output_dir = '/content/Yet-Another-EfficientDet-Pytorch/output_video_frames'
output_video_path = '/content/Yet-Another-EfficientDet-Pytorch/output_video.mp4'
# ✏️ 수정: 이전 실행의 프레임이 섞이지 않도록 폴더를 비우고 새로 만듦
import shutil
shutil.rmtree(output_dir, ignore_errors=True)
os.makedirs(output_dir)

compound_coef = 0
force_input_size = 512  # set None to use default size

threshold = 0.2
iou_threshold = 0.2

use_cuda = torch.cuda.is_available()  # ✏️ 수정: GPU 유무 자동 판단
use_float16 = False

obj_list = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
            'fire hydrant', '', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep',
            'cow', 'elephant', 'bear', 'zebra', 'giraffe', '', 'backpack', 'umbrella', '', '', 'handbag', 'tie',
            'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
            'skateboard', 'surfboard', 'tennis racket', 'bottle', '', 'wine glass', 'cup', 'fork', 'knife', 'spoon',
            'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut',
            'cake', 'chair', 'couch', 'potted plant', 'bed', '', 'dining table', '', '', 'toilet', '', 'tv',
            'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
            'refrigerator', '', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
            'toothbrush']

input_sizes = [512, 640, 768, 896, 1024, 1280, 1280, 1536, 1536]
input_size = input_sizes[compound_coef] if force_input_size is None else force_input_size

# load model
model = EfficientDetBackbone(compound_coef=compound_coef, num_classes=len(obj_list))
model.load_state_dict(torch.load(f'weights/efficientdet-d{compound_coef}.pth', map_location='cpu'))
model.requires_grad_(False)
model.eval();

In [ ]:
if use_cuda:
    model = model.cuda()
if use_float16:
    model = model.half()

In [ ]:
# function for display
def display(preds, imgs):
    for i in range(len(imgs)):
        if len(preds[i]['rois']) == 0:
            return imgs[i]

        for j in range(len(preds[i]['rois'])):
            (x1, y1, x2, y2) = preds[i]['rois'][j].astype(np.int64)
            cv2.rectangle(imgs[i], (x1, y1), (x2, y2), (255, 255, 0), 2)
            obj = obj_list[preds[i]['class_ids'][j]]
            score = float(preds[i]['scores'][j])

            cv2.putText(imgs[i], '{}, {:.3f}'.format(obj, score),
                        (x1, y1 + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                        (255, 255, 0), 1)

        return imgs[i]
# Box
regressBoxes = BBoxTransform()
clipBoxes = ClipBoxes()

In [ ]:
# Video capture
cap = cv2.VideoCapture(video_src)
fps = cap.get(cv2.CAP_PROP_FPS) or 20  # ✏️ 수정: 원본 영상 fps를 저장해 두었다가 영상 만들 때 사용
MAX_FRAMES = None                       # ✏️ 수정: 프레임 수 제한 옵션 (None = 전체, 느리면 300 등으로 줄이기)
frame_idx = 0
while MAX_FRAMES is None or frame_idx < MAX_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break

    # frame preprocessing
    ori_imgs, framed_imgs, framed_metas = preprocess_video(frame, max_size=input_size)

    if use_cuda:
        x = torch.stack([torch.from_numpy(fi).cuda() for fi in framed_imgs], 0)
    else:
        x = torch.stack([torch.from_numpy(fi) for fi in framed_imgs], 0)

    x = x.to(torch.float32 if not use_float16 else torch.float16).permute(0, 3, 1, 2)

    # model predict
    with torch.no_grad():
        features, regression, classification, anchors = model(x)

        out = postprocess(x,
                        anchors, regression, classification,
                        regressBoxes, clipBoxes,
                        threshold, iou_threshold)

    # result
    out = invert_affine(framed_metas, out)
    img_show = display(out, ori_imgs)  # ✏️ 수정: 같은 함수를 두 번 호출하던 중복 제거

    # 프레임 저장
    frame_path = os.path.join(output_dir, f'frame_{frame_idx:05d}.jpg')
    cv2.imwrite(frame_path, img_show)
    frame_idx += 1


cap.release()
print(f'저장된 프레임 수: {frame_idx}, fps: {fps}')
# cv2.destroyAllWindows()

In [ ]:
!ls -l

In [ ]:
os.listdir(output_dir)[:10]

In [ ]:
# ✏️ 수정: 파일명을 고정하지 않고 저장된 프레임 중 5번째를 표시
frames = sorted(os.listdir(output_dir))
img = Image.open(os.path.join(output_dir, frames[min(4, len(frames) - 1)]))
img

In [ ]:
# 🔄 이미지들을 비디오로 합치기
images = sorted([img for img in os.listdir(output_dir) if img.endswith(".jpg")])
first_frame = cv2.imread(os.path.join(output_dir, images[0]))
height, width, _ = first_frame.shape

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_video = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))  # ✏️ 수정: 원본 fps 사용

for image_file in images:
    img = cv2.imread(os.path.join(output_dir, image_file))
    out_video.write(img)

out_video.release()
print(f"✅ 영상 생성 완료: {output_video_path}")

In [ ]:
os.path.exists(output_video_path)

In [ ]:
# ✏️ 수정(추가): mp4v 코덱은 브라우저에서 재생이 안 되므로 H.264로 변환해서 노트북 안에서 바로 재생
from IPython.display import HTML
from base64 import b64encode
!ffmpeg -y -loglevel error -i {output_video_path} -vcodec libx264 -pix_fmt yuv420p output_video_h264.mp4
mp4 = open('output_video_h264.mp4', 'rb').read()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

### 학습

## Custom dataset 학습 (Car video)
https://www.kaggle.com/datasets/pkdarabi/vehicle-detection-image-dataset

### 1. 데이터셋 준비

- Yet-Another-EfficientDet-Pytorch
    - datasets
        - car_detect
            - Apply_Grayscale
                - Apply_Grayscale
                    - Vehicles_Detection.v9i.coco
                        - test
                            - _annotations.coco.json
                            - XXXX.jpg
                        - train
                        - valid
                

- Yet-Another-EfficientDet-Pytorch
    - projects
        - my_car_detect_proj.yml
    - datasets
        - car_detect
        - my_car_detect_proj
            - annotations
                - instances_train.json
                - instances_valid.json
                - instances_test.json
            - train
                - XXX.jpg
                - XXX.jpg
            - valid
                - XXX.jpg
                - XXX.jpg
            - test
                - XXX.jpg
                - XXX.jpg

### 2. yml 파일 준비

 - projects
     - my_car_detect_proj.yml

In [ ]:
import os
import json
import cv2
import matplotlib.pyplot as plt

def show_single_coco_image(
    coco_json_path,
    image_dir,
    image_index=0  # 시각화할 이미지 순서 (0부터 시작)
):
    # Load annotation JSON
    with open(coco_json_path, 'r') as f:
        coco = json.load(f)

    # Map image_id to image metadata
    image_info = coco['images'][image_index]
    image_id = image_info['id']
    file_name = image_info['file_name']
    img_path = os.path.join(image_dir, file_name)

    # Load image
    img = cv2.imread(img_path)
    if img is None:
        print(f"이미지를 불러올 수 없습니다: {img_path}")
        return

    # Index annotations for this image
    anns = [ann for ann in coco['annotations'] if ann['image_id'] == image_id]
    cat_id_to_name = {cat['id']: cat['name'] for cat in coco['categories']}

    # Draw bounding boxes
    for ann in anns:
        x, y, w, h = map(int, ann['bbox'])
        category_id = ann['category_id']
        label = cat_id_to_name.get(category_id, str(category_id))

        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(img, label, (x, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    # Convert BGR to RGB for matplotlib
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Show image
    plt.figure(figsize=(10, 10))
    plt.imshow(img_rgb)
    plt.title(file_name)
    plt.axis('off')
    plt.show()

In [ ]:
!ls -l

In [ ]:
!pwd

In [ ]:
# ✏️ 수정: archive.zip을 직접 올리는 대신 Kaggle에서 바로 내려받음 (공개 데이터셋, 로그인 불필요, 약 275MB)
os.makedirs('datasets', exist_ok=True)
if not os.path.exists('datasets/archive.zip'):
    !curl -L -o datasets/archive.zip https://www.kaggle.com/api/v1/datasets/download/pkdarabi/vehicle-detection-image-dataset
if not os.path.exists('datasets/car_detect'):
    !unzip -q datasets/archive.zip -d datasets/car_detect

# ✏️ 수정: 분석·학습·테스트에 쓸 데이터 버전을 하나로 통일 (원본은 분석=컬러 v8i, 학습=흑백 v9i로 섞여 있었음)
DATA_ROOT = 'datasets/car_detect/No_Apply_Grayscale/No_Apply_Grayscale/Vehicles_Detection.v8i.coco'
print(os.listdir(DATA_ROOT))

In [ ]:
show_single_coco_image(
    coco_json_path=f'{DATA_ROOT}/test/_annotations.coco.json',  # ✏️ 수정: DATA_ROOT 사용
    image_dir=f'{DATA_ROOT}/test',
    image_index=0  # 첫 번째 이미지 시각화
)

### mean, std, anchor 계산

In [ ]:
import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm

def compute_mean_std(image_dir, max_images=None):
    image_paths = sorted(glob(os.path.join(image_dir, '*.jpg')))
    if max_images:
        image_paths = image_paths[:max_images]

    means = []
    stds = []

    for path in tqdm(image_paths, desc="Computing mean/std"):
        img = cv2.imread(path)  # BGR
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img / 255.0
        means.append(np.mean(img, axis=(0, 1)))
        stds.append(np.std(img, axis=(0, 1)))

    mean = np.mean(means, axis=0)
    std = np.mean(stds, axis=0)

    print(f"✅ RGB mean: {mean.tolist()}")
    print(f"✅ RGB std : {std.tolist()}")
    return mean.tolist(), std.tolist()

In [ ]:
# ✏️ 수정: 결과를 변수에 저장해 두었다가 아래 yml 자동 생성에 사용
mean, std = compute_mean_std(f'{DATA_ROOT}/train', max_images=500)  # 샘플 500장 기준

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

def analyze_bbox_ratios_and_scales(coco_json_path, show_plots=True):

    with open(coco_json_path, 'r') as f:
        coco = json.load(f)

    # 바운딩 박스 정보 추출
    widths, heights, areas = [], [], []
    category_stats = defaultdict(list)  # 카테고리별 통계

    # 카테고리 이름 매핑
    cat_id_to_name = {cat['id']: cat['name'] for cat in coco['categories']}

    for ann in coco['annotations']:
        x, y, w, h = ann['bbox']
        if w > 0 and h > 0:  # 유효한 바운딩 박스만
            widths.append(w)
            heights.append(h)
            areas.append(w * h)

            # 카테고리별 분석을 위해
            cat_name = cat_id_to_name.get(ann['category_id'], 'unknown')
            category_stats[cat_name].append((w, h))

    widths = np.array(widths)
    heights = np.array(heights)
    areas = np.array(areas)

    # 1. Aspect Ratios 계산
    ratios = widths / heights

    # 2. Scales 계산 (면적의 제곱근)
    scales = np.sqrt(areas)

    # 3. 시각화
    if show_plots:
        plt.figure(figsize=(15, 5))

        # Aspect Ratios 히스토그램
        plt.subplot(1, 3, 1)
        plt.hist(ratios, bins=50, alpha=0.7, edgecolor='black')
        plt.axvline(np.median(ratios), color='red', linestyle='--',
                   label=f'Median: {np.median(ratios):.2f}')
        plt.xlabel('Aspect Ratio (W/H)')
        plt.ylabel('Frequency')
        plt.title('Distribution of Aspect Ratios')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Scales 히스토그램
        plt.subplot(1, 3, 2)
        plt.hist(scales, bins=50, alpha=0.7, edgecolor='black')
        plt.axvline(np.median(scales), color='red', linestyle='--',
                   label=f'Median: {np.median(scales):.1f}')
        plt.xlabel('Scale (sqrt of area)')
        plt.ylabel('Frequency')
        plt.title('Distribution of Scales')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Log scale로 스케일 분포 확인
        plt.subplot(1, 3, 3)
        plt.hist(np.log(scales), bins=50, alpha=0.7, edgecolor='black')
        plt.xlabel('Log Scale')
        plt.ylabel('Frequency')
        plt.title('Distribution of Log Scales')
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    # 4. 통계 정보 출력
    print("📊 데이터셋 통계")
    print(f"총 annotation 수: {len(widths)}")
    print(f"평균 크기: {np.mean(scales):.1f}")
    print(f"중앙값 크기: {np.median(scales):.1f}")
    print(f"평균 비율: {np.mean(ratios):.2f}")
    print(f"중앙값 비율: {np.median(ratios):.2f}")
    print()

    # 5. Anchor Ratios 추천 (20%, 50%, 80% percentile)
    ratio_percentiles = np.percentile(ratios, [20, 50, 80])
    print("🎯 추천 Anchor Ratios")
    print("Percentile 기반 (20%, 50%, 80%):")

    # 단일 값 형태 (EfficientDet/YOLO 스타일)
    ratios_simple = [round(r, 2) for r in ratio_percentiles]
    print(f"  Simple format: {ratios_simple}")

    # Tuple 형태 (일부 구현체용)
    ratios_tuple = [(1.0, round(1/r, 2)) if r > 1 else (round(r, 2), 1.0)
                   for r in ratio_percentiles]
    print(f"  Tuple format: {ratios_tuple}")
    print()

    # 6. Anchor Scales 추천
    scale_percentiles = np.percentile(scales, [20, 50, 80])
    base_scale = scale_percentiles[1]  # 중앙값을 기준으로
    scales_normalized = [round(s / base_scale, 2) for s in scale_percentiles]

    print("📏 추천 Anchor Scales")
    print(f"절대값: {[round(s, 1) for s in scale_percentiles]}")
    print(f"정규화 (중앙값 기준): {scales_normalized}")
    print(f"기준 크기: {base_scale:.1f}")
    print()

    # 7. 실제 사용 예시
    print("💡 실제 설정 예시")
    print("# EfficientDet/RetinaNet 스타일")
    print(f"anchor_ratios = {ratios_simple}")
    print(f"anchor_scales = {scales_normalized}")
    print("")
    print("# 또는 전통적인 설정")
    print("anchor_ratios = [0.5, 1.0, 2.0]")
    print("anchor_scales = [1.0, 1.26, 1.587]  # 2^(0/3), 2^(1/3), 2^(2/3)")
    print()

    # 8. 카테고리별 분석 (상위 5개)
    print("📋 주요 카테고리별 비율 분석")
    cat_ratios = {}
    for cat_name, boxes in category_stats.items():
        if len(boxes) >= 10:  # 충분한 샘플이 있는 카테고리만
            boxes = np.array(boxes)
            cat_ratios[cat_name] = np.median(boxes[:, 0] / boxes[:, 1])

    # 빈도 기준 정렬
    sorted_cats = sorted(cat_ratios.items(),
                        key=lambda x: len(category_stats[x[0]]),
                        reverse=True)[:5]

    for cat_name, median_ratio in sorted_cats:
        count = len(category_stats[cat_name])
        print(f"  {cat_name}: {median_ratio:.2f} (samples: {count})")

    return {
        'ratios_simple': ratios_simple,
        'ratios_tuple': ratios_tuple,
        'scales_normalized': scales_normalized,
        'scales_absolute': [round(s, 1) for s in scale_percentiles],
        'base_scale': base_scale,
        'stats': {
            'total_annotations': len(widths),
            'mean_ratio': np.mean(ratios),
            'median_ratio': np.median(ratios),
            'mean_scale': np.mean(scales),
            'median_scale': np.median(scales)
        }
    }

In [ ]:
# ✏️ 수정: 결과를 변수에 저장해 두었다가 아래 yml 자동 생성에 사용
anchor_result = analyze_bbox_ratios_and_scales(f'{DATA_ROOT}/train/_annotations.coco.json')

In [ ]:
import json

def extract_obj_list(coco_json_path):
    with open(coco_json_path, 'r') as f:
        coco = json.load(f)

    obj_list = [cat['name'] for cat in sorted(coco['categories'], key=lambda x: x['id'])]
    print("obj_list:", obj_list)
    return obj_list

In [ ]:
all_names = extract_obj_list(f'{DATA_ROOT}/train/_annotations.coco.json')

# ✏️ 수정: 맨 앞 'cars'(id 0)는 Roboflow가 넣은 상위 카테고리로 실제 박스가 0개.
#          이 저장소는 category_id - 1 을 라벨로 쓰므로(Bus=id1 → 라벨0) 'cars'를 빼야 이름이 맞음
custom_obj_list = all_names[1:]
print("실제 사용할 obj_list:", custom_obj_list)

- Yet-Another-EfficientDet-Pytorch
    - projects
        - my_car_detect_proj.yml
    - datasets
        - car_detect
        - my_car_detect
            - annotations
                - instances_train.json
                - instances_valid.json
                - instances_test.json
            - train
                - XXX.jpg
                - XXX.jpg
            - valid
                - XXX.jpg
                - XXX.jpg
            - test
                - XXX.jpg
                - XXX.jpg

In [ ]:
import os
import shutil
import json
from pathlib import Path

# 1) 설정: 원본/대상 경로 및 split 정의
SRC_ROOT = Path(DATA_ROOT)  # ✏️ 수정: 분석한 데이터와 같은 버전(컬러 v8i)으로 학습
DST_ROOT = Path("datasets/my_car_detect_proj")
ANNOT_DIR = DST_ROOT / "annotations"
SPLITS = ["train", "valid", "test"]

# 2) 대상 디렉토리 생성
for p in [DST_ROOT, ANNOT_DIR] + [DST_ROOT / s for s in SPLITS]:
    p.mkdir(parents=True, exist_ok=True)

# 3) split별 이미지 복사 및 annotation 변환·저장
for split in SPLITS:
    src_split = SRC_ROOT / split
    dst_split = DST_ROOT / split
    dst_split.mkdir(exist_ok=True)

    # 3-1) 이미지 파일(.jpg/.png) 복사
    for ext in ("*.jpg", "*.png"):
        for img_path in src_split.glob(ext):
            shutil.copy(img_path, dst_split / img_path.name)

    # 3-2) annotation JSON 변환
    #    - 원본 JSON 불러오기
    #    - COCO to EfficientDet-PyTorch 형식 변환
    #    - instances_{split}.json으로 저장
    for json_path in src_split.glob("*.json"):
        with open(json_path, "r", encoding="utf-8") as f:
            before = json.load(f)

        # 새 구조 준비
        tobe = {
            "images": [],
            "type": "instances",
            "annotations": [],
            "categories": []
        }

        # images 필드
        for img in before.get("images", []):
            tobe["images"].append({
                "file_name": img["file_name"],
                "height": img["height"],
                "width": img["width"],
                "id": img["id"]
            })

        # annotations 필드
        for ann in before.get("annotations", []):
            tobe["annotations"].append({
                "id": ann["id"],
                "image_id": ann["image_id"],
                "category_id": ann["category_id"],
                "bbox": ann["bbox"],
                "area": ann["area"],
                "iscrowd": ann.get("iscrowd", 0),
                "ignore": 0,
                "segmentation": ann.get("segmentation", [])
            })

        # categories 필드
        for cat in before.get("categories", []):
            tobe["categories"].append({
                "supercategory": cat.get("supercategory", ""),
                "id": cat["id"],
                "name": cat["name"]
            })

        # 저장
        dst_json = ANNOT_DIR / f"instances_{split}.json"
        with open(dst_json, "w", encoding="utf-8") as f:
            json.dump(tobe, f, ensure_ascii=False, indent=2)

        print(f"[{split}] {json_path.name} → {dst_json.name}")

print("✅ 전체 폴더 구조 및 annotation 변환 완료!")

> ✏️ 수정: 원본의 "폴더 구조 변환(단순 복사)" 셀은 바로 위에서 변환한 json을 원본으로 **덮어써서** 삭제했습니다.

### 2. yml 파일 준비

In [ ]:
# ✏️ 수정: 원본은 yml 파일을 직접 만들어 올려야 했음 → 위에서 계산한 값으로 자동 생성
import torch
ratios = [(round(float(w), 2), round(float(h), 2)) for w, h in anchor_result['ratios_tuple']]
scales = [round(float(s), 2) for s in anchor_result['scales_normalized']]

yml = f"""project_name: my_car_detect_proj  # datasets/ 아래 데이터 폴더 이름과 같아야 함
train_set: train
val_set: valid
num_gpus: {1 if torch.cuda.is_available() else 0}  # 0이면 train.py가 GPU를 끄고 CPU로 학습함

# mean and std in RGB order
mean: {[round(float(m), 4) for m in mean]}
std: {[round(float(s), 4) for s in std]}

# 데이터 분석으로 구한 앵커 (학습·평가·추론 모두 이 값을 사용)
anchors_scales: '{scales}'
anchors_ratios: '{ratios}'

# category_id - 1 순서와 일치해야 함
obj_list: {custom_obj_list}
"""
os.makedirs('projects', exist_ok=True)
with open('projects/my_car_detect_proj.yml', 'w') as f:
    f.write(yml)
print(yml)

## 학습

In [ ]:
# ✏️ 수정: 최신 PyTorch에는 ReduceLROnPlateau의 verbose 인자가 없어서 에러 → 원본 안내대로 자동 수정
!sed -i 's/patience=3, verbose=True)/patience=3)/' train.py
!grep -n "ReduceLROnPlateau" train.py

In [ ]:
!pwd

In [ ]:
# 1단계: head만 학습
# ✏️ 수정: -n 2 추가 (Colab CPU 코어 2개에 맞춰 DataLoader worker 수 조정, 경고·멈춤 방지)
!python train.py -c 0 -p my_car_detect_proj --head_only True --lr 1e-3 --batch_size 16 --load_weights weights/efficientdet-d0.pth --num_epochs 10 --save_interval 100 -n 2

In [ ]:
# 2단계: 전체 fine-tuning
# ✏️ 수정: --load_weights last → 파일명을 직접 적지 않아도 가장 최근 가중치를 자동으로 불러옴
#          lr 1e-3 → 1e-4 (backbone까지 학습할 때는 작은 학습률이 안정적)
!python train.py -c 0 -p my_car_detect_proj --head_only False --lr 1e-4 --batch_size 16 --load_weights last --num_epochs 30 --save_interval 100 -n 2

In [ ]:
# ✏️ 수정: %cd로 폴더를 옮기지 않고 가장 최근 가중치 경로만 찾음 (이후 경로가 꼬이지 않음)
from glob import glob
weight_files = sorted(glob('logs/my_car_detect_proj/*.pth'), key=os.path.getmtime)
weight_path = weight_files[-1]
print(len(weight_files), "개 중 최근 가중치:", weight_path)

In [ ]:
# ✏️ 수정: weight_path 사용, GPU 없을 때도 동작하도록 --cuda 지정
!python coco_eval.py -c 0 -p my_car_detect_proj -w "{weight_path}" --cuda {use_cuda}

In [ ]:
# ✏️ 수정: DATA_ROOT 기준 경로, 파일이 없으면 test 폴더의 첫 이미지 사용
img_path = f'{DATA_ROOT}/test/frame_0990_jpg.rf.a3223422e734a57442ee34a58d24d4b4.jpg'
if not os.path.exists(img_path):
    img_path = sorted(glob(f'{DATA_ROOT}/test/*.jpg'))[0]
print(img_path)

In [ ]:
import torch
from torch.backends import cudnn

from backbone import EfficientDetBackbone
import cv2
import matplotlib.pyplot as plt
import numpy as np

from efficientdet.utils import BBoxTransform, ClipBoxes
from utils.utils import preprocess, invert_affine, postprocess

compound_coef = 0
force_input_size = None  # set None to use default size
import yaml
# ✏️ 수정: 학습에 쓴 yml을 그대로 읽어서 클래스·앵커·mean/std를 학습과 똑같이 맞춤
params = yaml.safe_load(open('projects/my_car_detect_proj.yml'))
# img_path는 바로 위 셀에서 정함

threshold = 0.4
iou_threshold = 0.2

use_cuda = torch.cuda.is_available()  # ✏️ 수정: GPU 유무 자동 판단
use_float16 = False

obj_list = params['obj_list']  # ✏️ 수정: 'cars' 없는 5개 클래스

# tf bilinear interpolation is different from any other's, just make do
input_sizes = [512, 640, 768, 896, 1024, 1280, 1280, 1536]
input_size = input_sizes[compound_coef] if force_input_size is None else force_input_size
ori_imgs, framed_imgs, framed_metas = preprocess(img_path, max_size=input_size, mean=params['mean'], std=params['std'])  # ✏️ 수정: 학습 때와 같은 정규화

if use_cuda:
    x = torch.stack([torch.from_numpy(fi).cuda() for fi in framed_imgs], 0)
else:
    x = torch.stack([torch.from_numpy(fi) for fi in framed_imgs], 0)

x = x.to(torch.float32 if not use_float16 else torch.float16).permute(0, 3, 1, 2)

model = EfficientDetBackbone(compound_coef=compound_coef, num_classes=len(obj_list),

                             # ✏️ 수정: 학습(yml)과 다른 앵커를 쓰던 문제 → yml 값 사용
                             ratios=eval(params['anchors_ratios']),
                             scales=eval(params['anchors_scales']))

model.load_state_dict(torch.load(weight_path, map_location='cpu'))  # ✏️ 수정: 위에서 찾은 가중치
model.requires_grad_(False)
model.eval()

if use_cuda:
    model = model.cuda()
if use_float16:
    model = model.half()

with torch.no_grad():
    features, regression, classification, anchors = model(x)

    regressBoxes = BBoxTransform()
    clipBoxes = ClipBoxes()

    out = postprocess(x,
                      anchors, regression, classification,
                      regressBoxes, clipBoxes,
                      threshold, iou_threshold)

out = invert_affine(framed_metas, out)

for i in range(len(ori_imgs)):
    if len(out[i]['rois']) == 0:
        continue
    ori_imgs[i] = ori_imgs[i].copy()
    for j in range(len(out[i]['rois'])):
        (x1, y1, x2, y2) = out[i]['rois'][j].astype(np.int32)
        cv2.rectangle(ori_imgs[i], (x1, y1), (x2, y2), (255, 255, 0), 2)
        obj = obj_list[out[i]['class_ids'][j]]
        score = float(out[i]['scores'][j])

        cv2.putText(ori_imgs[i], '{}, {:.3f}'.format(obj, score),
                    (x1, y1 + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (255, 255, 0), 1)

    # ✏️ 수정: 박스마다 imshow 하던 것을 다 그린 뒤 한 번만 표시, OpenCV(BGR) → RGB 변환
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(ori_imgs[i], cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

print("검출된 박스 수:", len(out[0]['rois']))

In [ ]:
out